# Train-wise ETA/Delay Prediction — CatBoost vs LSTM vs GRU

Google Colab notebook. Upload the CSV when prompted.

The notebook first inspects the uploaded dataset, detects the actual columns, cleans common inconsistencies, creates leakage-safe features, clips extreme numeric values using training-only quantiles, then trains a separate CatBoost, LSTM and GRU model for every train and compares MAE, RMSE and R².


In [15]:
# 1. IMPORTS
!pip -q install catboost

import io
import re
import random
import warnings
import numpy as np
import pandas as pd

from google.colab import files
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("All imports loaded successfully.")


All imports loaded successfully.


## 2. Upload and inspect the dataset

In [17]:
uploaded = files.upload()
if not uploaded:
    raise ValueError("No dataset was uploaded.")

file_name = next(iter(uploaded))
df = pd.read_excel(io.BytesIO(uploaded[file_name]))

print("File:", file_name)
print("Shape:", df.shape)
print("\nColumns:")
for i, c in enumerate(df.columns, 1):
    print(f"{i:2d}. {c}")

display(df.head())
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))

Saving fffgfggcccgcg_combined_v3.xlsx to fffgfggcccgcg_combined_v3 (1).xlsx
File: fffgfggcccgcg_combined_v3 (1).xlsx
Shape: (44366, 21)

Columns:
 1. journey_id
 2. train_no
 3. date
 4. route_id
 5. station_sequence
 6. station_code
 7. station_name
 8. distance
 9. previous_station
10. perivious distance
11. section_distance
12. delay
13. previous_delay
14. delay_change
15. journey_progress
16. total_stations
17. day_of_week
18. is_weekend
19. month
20. source_file
21. day


,journey_id,train_no,date,route_id,station_sequence,station_code,station_name,distance,previous_station,perivious distance,...,delay,previous_delay,delay_change,journey_progress,total_stations,day_of_week,is_weekend,month,source_file,day
0,12314_20250905,12314,2025-09-05,12314_NDLS_SDAH,0,NDLS,NEW DELHI,0,START,0,...,0.0,0.0,0.0,0.000000,8,4,0,9,12314_running_history.csv,5
1,12314_20250905,12314,2025-09-05,12314_NDLS_SDAH,1,CNB,KANPUR CENTRAL,440,NDLS,0,...,4.0,0.0,4.0,0.142857,8,4,0,9,12314_running_history.csv,5
2,12314_20250905,12314,2025-09-05,12314_NDLS_SDAH,2,DDU,PT DEEN DAYAL UPADHYAY JN,787,CNB,440,...,0.0,4.0,-4.0,0.285714,8,4,0,9,12314_running_history.csv,5
3,12314_20250905,12314,2025-09-05,12314_NDLS_SDAH,3,GAYA,GAYA JN,990,DDU,787,...,13.0,0.0,13.0,0.428571,8,4,0,9,12314_running_history.csv,5
4,12314_20250905,12314,2025-09-05,12314_NDLS_SDAH,4,DHN,DHANBAD JN,1189,GAYA,990,...,2.0,13.0,-11.0,0.571429,8,4,0,9,12314_running_history.csv,5


,dtype
journey_id,object
train_no,int64
date,datetime64[ns]
route_id,object
station_sequence,int64
station_code,object
station_name,object
distance,int64
previous_station,object
perivious distance,int64


,missing
journey_id,0
train_no,0
date,0
route_id,0
station_sequence,0
station_code,0
station_name,0
distance,0
previous_station,0
perivious distance,0


## 3. Normalize column names and identify the target

In [18]:
def norm_col(c):
    c = str(c).strip().lower()
    c = re.sub(r"[^a-z0-9]+", "_", c)
    return c.strip("_")

data = df.rename(columns={c: norm_col(c) for c in df.columns}).copy()

# Common aliases found in train datasets.
alias_groups = {
    "train_no": ["train_no", "train_number", "train_num", "train"],
    "date": ["date", "journey_date", "running_date"],
    "station_code": ["station_code", "stn_code", "station"],
    "station_name": ["station_name", "stn_name"],
    "current_delay": ["current_delay", "current_delay_minutes"], # Removed 'delay'
    "previous_delay": ["previous_delay", "perivious_delay", "prev_delay"],
    "section_distance": ["section_distance", "perivious_distance", "previous_distance", "distance"],
    "final_delay": ["final_delay", "destination_delay", "target_delay", "final_delay_minutes", "delay"] # Added 'delay'
}

rename = {}
for canonical, aliases in alias_groups.items():
    if canonical not in data.columns:
        for a in aliases:
            if a in data.columns:
                rename[a] = canonical
                break

data = data.rename(columns=rename)

print("Normalized/standardized columns:")
print(list(data.columns))

if "train_no" not in data.columns:
    raise ValueError("Train number column was not found.")

if "final_delay" not in data.columns:
    raise ValueError(
        "Target column was not found. Expected final_delay, "
        "destination_delay or target_delay."
    )

print("\nTrain count:", data["train_no"].nunique())
print("Trains:", data["train_no"].drop_duplicates().tolist())

Normalized/standardized columns:
['journey_id', 'train_no', 'date', 'route_id', 'station_sequence', 'station_code', 'station_name', 'distance', 'previous_station', 'perivious_distance', 'section_distance', 'final_delay', 'previous_delay', 'delay_change', 'journey_progress', 'total_stations', 'day_of_week', 'is_weekend', 'month', 'source_file', 'day']

Train count: 8
Trains: [12314, 12428, 12628, 12802, 12958, 14310, 22457, 22487]


## 4. Clean data and engineer features

In [19]:
# Date
if "date" in data.columns:
    data["date"] = pd.to_datetime(data["date"], errors="coerce")

# Convert likely numeric columns
for c in data.columns:
    if c not in ["train_no", "station_code", "station_name", "date"]:
        data[c] = pd.to_numeric(data[c], errors="coerce")

data = data.replace([np.inf, -np.inf], np.nan)
data = data.dropna(subset=["final_delay"]).copy()

# Sort first so lag features use past observations only.
sort_cols = ["train_no"]
if "date" in data.columns:
    sort_cols.append("date")
data = data.sort_values(sort_cols).reset_index(drop=True)

# Date-derived features
if "date" in data.columns:
    data["year"] = data["date"].dt.year
    data["month"] = data["date"].dt.month
    data["day"] = data["date"].dt.day
    data["day_of_week"] = data["date"].dt.dayofweek
    data["day_of_year"] = data["date"].dt.dayofyear

# Make lag features within each train + station.
group_cols = ["train_no"]
if "station_code" in data.columns:
    group_cols.append("station_code")
elif "station_name" in data.columns:
    group_cols.append("station_name")

if "date" in data.columns:
    data["calculated_previous_delay"] = (
        data.groupby(group_cols, dropna=False)["final_delay"].shift(1)
    )

if "current_delay" in data.columns and "calculated_previous_delay" in data.columns:
    data["delay_change"] = (
        data["current_delay"] - data["calculated_previous_delay"]
    )

if "previous_delay" in data.columns:
    if "calculated_previous_delay" in data.columns:
        data["previous_delay_used"] = (
            data["calculated_previous_delay"]
            .fillna(data["previous_delay"])
        )
    else:
        data["previous_delay_used"] = data["previous_delay"]
elif "calculated_previous_delay" in data.columns:
    data["previous_delay_used"] = data["calculated_previous_delay"]

# Never use target or obvious target copies as input.
target_like = [
    "final_delay", "destination_delay", "target_delay",
    "actual_arrival", "arrival_delay_at_destination", "eta_actual"
]

categorical_cols = [
    c for c in ["station_code", "station_name"]
    if c in data.columns
]

numeric_cols = []
for c in data.columns:
    if c in target_like or c in ["date", "train_no"] or c in categorical_cols:
        continue
    if pd.api.types.is_numeric_dtype(data[c]):
        numeric_cols.append(c)

feature_cols = categorical_cols + numeric_cols

# Drop unusable all-null numeric columns.
numeric_cols = [c for c in numeric_cols if data[c].notna().any()]
feature_cols = categorical_cols + numeric_cols

for c in categorical_cols:
    data[c] = data[c].astype("string").fillna("UNKNOWN")

print("Categorical features:", categorical_cols)
print("Numeric features:", numeric_cols)
print("Total features:", len(feature_cols))
display(data[feature_cols + ["final_delay"]].head())


Categorical features: ['station_code', 'station_name']
Numeric features: ['station_sequence', 'distance', 'perivious_distance', 'section_distance', 'previous_delay', 'delay_change', 'journey_progress', 'total_stations', 'day_of_week', 'is_weekend', 'month', 'day', 'year', 'day_of_year', 'calculated_previous_delay', 'previous_delay_used']
Total features: 18


,station_code,station_name,station_sequence,distance,perivious_distance,section_distance,previous_delay,delay_change,journey_progress,total_stations,day_of_week,is_weekend,month,day,year,day_of_year,calculated_previous_delay,previous_delay_used,final_delay
0,NDLS,NEW DELHI,0,0,0,0,0.0,0.0,0.000000,8,4,0,9,5,2025,248,NaN,0.0,0.0
1,CNB,KANPUR CENTRAL,1,440,0,440,0.0,4.0,0.142857,8,4,0,9,5,2025,248,NaN,0.0,4.0
2,DDU,PT DEEN DAYAL UPADHYAY JN,2,787,440,347,4.0,-4.0,0.285714,8,4,0,9,5,2025,248,NaN,4.0,0.0
3,GAYA,GAYA JN,3,990,787,203,0.0,13.0,0.428571,8,4,0,9,5,2025,248,NaN,0.0,13.0
4,DHN,DHANBAD JN,4,1189,990,199,13.0,-11.0,0.571429,8,4,0,9,5,2025,248,NaN,13.0,2.0


## 5. Train/test and outlier helpers

In [20]:
def chronological_split(frame, test_fraction=0.20):
    frame = frame.copy()
    if "date" in frame.columns:
        frame = frame.sort_values("date")
    n = len(frame)
    cut = max(1, int(n * (1 - test_fraction)))
    cut = min(cut, n - 1)
    return frame.iloc[:cut].copy(), frame.iloc[cut:].copy()

def clip_by_train_quantiles(X_train, X_test, cols, low=0.01, high=0.99):
    X_train = X_train.copy()
    X_test = X_test.copy()
    for c in cols:
        if c not in X_train.columns:
            continue
        lo = X_train[c].quantile(low)
        hi = X_train[c].quantile(high)
        if pd.notna(lo) and pd.notna(hi) and lo < hi:
            X_train[c] = X_train[c].clip(lo, hi)
            X_test[c] = X_test[c].clip(lo, hi)
    return X_train, X_test

def get_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    }


## 6. CatBoost — one separate model per train

In [28]:
def run_catboost(frame, train_no):
    tr, te = chronological_split(frame)

    Xtr = tr[feature_cols].copy()
    Xte = te[feature_cols].copy()
    ytr = tr["final_delay"].astype(float)
    yte = te["final_delay"].astype(float)

    Xtr, Xte = clip_by_train_quantiles(Xtr, Xte, numeric_cols)

    for c in numeric_cols:
        med = Xtr[c].median()
        if pd.isna(med):
            med = 0.0
        Xtr[c] = Xtr[c].fillna(med)
        Xte[c] = Xte[c].fillna(med)

    for c in categorical_cols:
        Xtr[c] = Xtr[c].astype(str)
        Xte[c] = Xte[c].astype(str)

    model = CatBoostRegressor(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        loss_function="RMSE",
        random_seed=SEED,
        verbose=False
    )

    model.fit(Xtr, ytr, cat_features=categorical_cols)

    pred = model.predict(Xte)
    m = get_metrics(yte, pred)

    pred_out = te.copy()
    pred_out["prediction"] = pred
    pred_out["model"] = "CatBoost"
    pred_out["train_no"] = train_no

    importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.get_feature_importance()
    }).sort_values("importance", ascending=False)

    return m, pred_out, importance, model

## 7. Sequence preprocessing — no station_code indexing after one-hot encoding

In [22]:
def sequence_arrays(tr, te):
    Xtr = tr[feature_cols].copy()
    Xte = te[feature_cols].copy()

    Xtr, Xte = clip_by_train_quantiles(Xtr, Xte, numeric_cols)

    for c in categorical_cols:
        Xtr[c] = Xtr[c].astype(str).fillna("UNKNOWN")
        Xte[c] = Xte[c].astype(str).fillna("UNKNOWN")

    for c in numeric_cols:
        med = Xtr[c].median()
        if pd.isna(med):
            med = 0.0
        Xtr[c] = Xtr[c].fillna(med)
        Xte[c] = Xte[c].fillna(med)

    # Encode categories. After this point we NEVER try to access
    # X["station_code"], so the previous KeyError cannot occur.
    combined = pd.concat([Xtr, Xte], ignore_index=True)
    combined = pd.get_dummies(
        combined,
        columns=categorical_cols,
        dtype=np.float32
    )

    Xtr = combined.iloc[:len(Xtr)].to_numpy(dtype=np.float32)
    Xte = combined.iloc[len(Xtr):].to_numpy(dtype=np.float32)

    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr).astype(np.float32)
    Xte = scaler.transform(Xte).astype(np.float32)

    return Xtr, Xte

def make_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i-seq_len:i])
        ys.append(y[i])
    return np.asarray(Xs, dtype=np.float32), np.asarray(ys, dtype=np.float32)


In [23]:
def build_lstm(input_shape):
    model = Sequential([
        LSTM(64, input_shape=input_shape),
        Dropout(0.20),
        Dense(32, activation="relu"),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

def build_gru(input_shape):
    model = Sequential([
        GRU(64, input_shape=input_shape),
        Dropout(0.20),
        Dense(32, activation="relu"),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model


In [24]:
def run_sequence_model(frame, train_no, kind, seq_len=5):
    tr, te = chronological_split(frame)

    if len(tr) <= seq_len or len(te) == 0:
        raise ValueError(
            f"Not enough rows for {kind}: train={len(tr)}, test={len(te)}"
        )

    Xtr, Xte = sequence_arrays(tr, te)
    ytr = tr["final_delay"].astype(float).to_numpy()
    yte = te["final_delay"].astype(float).to_numpy()

    X_train_seq, y_train_seq = make_sequences(Xtr, ytr, seq_len)

    # Add the last training observations as context for the first test prediction.
    context_X = np.vstack([Xtr[-seq_len:], Xte])
    context_y = np.concatenate([ytr[-seq_len:], yte])

    X_test_seq, y_test_seq = make_sequences(
        context_X, context_y, seq_len
    )

    if kind == "LSTM":
        model = build_lstm((X_train_seq.shape[1], X_train_seq.shape[2]))
    else:
        model = build_gru((X_train_seq.shape[1], X_train_seq.shape[2]))

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )

    model.fit(
        X_train_seq,
        y_train_seq,
        epochs=60,
        batch_size=32,
        validation_split=0.15,
        shuffle=False,
        callbacks=[early_stop],
        verbose=0
    )

    pred = model.predict(X_test_seq, verbose=0).ravel()
    m = get_metrics(y_test_seq, pred)

    pred_out = te.iloc[:len(pred)].copy()
    pred_out["prediction"] = pred
    pred_out["model"] = kind
    pred_out["train_no"] = train_no

    return m, pred_out


## 8. Train CatBoost, LSTM and GRU separately for every train

In [29]:
results = []
predictions = []
feature_importance = {}
catboost_models = {}

for train_no in data["train_no"].dropna().unique():
    train_data = data[data["train_no"] == train_no].copy()

    if len(train_data) < 30:
        print(f"Skipping train {train_no}: only {len(train_data)} rows")
        continue

    print(f"\n===== Train {train_no}: {len(train_data)} rows =====")

    try:
        m, p, imp, model = run_catboost(train_data, train_no)
        results.append({"train_no": train_no, "model": "CatBoost", **m})
        predictions.append(p)
        feature_importance[train_no] = imp
        catboost_models[train_no] = model # Store the trained model
        print(f"CatBoost | MAE={m['MAE']:.3f} RMSE={m['RMSE']:.3f} R2={m['R2']:.3f}")
    except Exception as e:
        print("CatBoost failed:", repr(e))

    try:
        m, p = run_sequence_model(train_data, train_no, "LSTM", seq_len=5)
        results.append({"train_no": train_no, "model": "LSTM", **m})
        predictions.append(p)
        print(f"LSTM     | MAE={m['MAE']:.3f} RMSE={m['RMSE']:.3f} R2={m['R2']:.3f}")
    except Exception as e:
        print("LSTM failed:", repr(e))

    try:
        m, p = run_sequence_model(train_data, train_no, "GRU", seq_len=5)
        results.append({"train_no": train_no, "model": "GRU", **m})
        predictions.append(p)
        print(f"GRU      | MAE={m['MAE']:.3f} RMSE={m['RMSE']:.3f} R2={m['R2']:.3f}")
    except Exception as e:
        print("GRU failed:", repr(e))


===== Train 12314: 2920 rows =====
CatBoost | MAE=1.588 RMSE=3.112 R2=0.989
LSTM     | MAE=18.369 RMSE=29.492 R2=-0.005
GRU      | MAE=13.664 RMSE=24.717 R2=0.294

===== Train 12428: 5110 rows =====
CatBoost | MAE=1.494 RMSE=3.257 R2=0.992
LSTM     | MAE=12.078 RMSE=22.125 R2=0.609
GRU      | MAE=13.520 RMSE=22.798 R2=0.585

===== Train 12628: 12410 rows =====
CatBoost | MAE=1.858 RMSE=5.628 R2=0.981
LSTM     | MAE=15.972 RMSE=23.438 R2=0.677
GRU      | MAE=16.921 RMSE=24.461 R2=0.648

===== Train 12802: 12045 rows =====
CatBoost | MAE=2.354 RMSE=8.766 R2=0.957
LSTM     | MAE=15.560 RMSE=25.336 R2=0.644
GRU      | MAE=15.104 RMSE=23.993 R2=0.681

===== Train 12958: 3650 rows =====
CatBoost | MAE=0.779 RMSE=2.716 R2=0.977
LSTM     | MAE=6.396 RMSE=10.724 R2=0.648
GRU      | MAE=5.969 RMSE=10.373 R2=0.671

===== Train 14310: 3536 rows =====
CatBoost | MAE=5.405 RMSE=13.111 R2=0.968
LSTM     | MAE=20.508 RMSE=34.969 R2=0.771
GRU      | MAE=19.205 RMSE=33.807 R2=0.786

===== Train 22457: 

## 9. Compare all three models

In [26]:
results_df = pd.DataFrame(results)

if results_df.empty:
    raise ValueError("No models completed successfully.")

results_df = results_df.sort_values(["train_no", "MAE"]).reset_index(drop=True)

print("Per-train results:")
display(results_df)

print("\nAverage performance across trains:")
overall = (
    results_df.groupby("model")[["MAE", "RMSE", "R2"]]
    .mean()
    .sort_values("MAE")
)
display(overall)

print("\nBest model for each train:")
best = (
    results_df.sort_values(["train_no", "MAE"])
    .groupby("train_no", as_index=False)
    .first()
)
display(best)


Per-train results:


,train_no,model,MAE,RMSE,R2
0,12314,CatBoost,1.588427,3.111522,0.988812
1,12314,LSTM,13.795886,24.773574,0.290753
2,12314,GRU,17.114965,29.720368,-0.020771
3,12428,CatBoost,1.494028,3.257066,0.991534
4,12428,LSTM,12.102648,21.839101,0.619360
5,12428,GRU,12.878736,21.935423,0.615995
6,12628,CatBoost,1.857582,5.627655,0.981354
7,12628,GRU,15.937822,24.510714,0.646291
8,12628,LSTM,16.570990,25.513852,0.616746
9,12802,CatBoost,2.354068,8.766192,0.957422



Average performance across trains:


,MAE,RMSE,R2
model,,,
CatBoost,1.965157,5.880452,0.950394
LSTM,11.891551,20.146486,0.570360
GRU,12.241728,20.560037,0.544081



Best model for each train:


,train_no,model,MAE,RMSE,R2
0,12314,CatBoost,1.588427,3.111522,0.988812
1,12428,CatBoost,1.494028,3.257066,0.991534
2,12628,CatBoost,1.857582,5.627655,0.981354
3,12802,CatBoost,2.354068,8.766192,0.957422
4,12958,CatBoost,0.779347,2.716206,0.977442
5,14310,CatBoost,5.405003,13.111419,0.967763
6,22457,CatBoost,1.647305,8.472726,0.783863
7,22487,CatBoost,0.595492,1.980830,0.954960


In [27]:
# CatBoost feature importance
for train_no, imp in feature_importance.items():
    print(f"\nTop features for train {train_no}")
    display(imp.head(15))

# Save results
results_df.to_csv("model_comparison_per_train.csv", index=False)

if predictions:
    predictions_df = pd.concat(predictions, ignore_index=True)
    predictions_df.to_csv("model_predictions_per_train.csv", index=False)

print("Saved model_comparison_per_train.csv")
if predictions:
    print("Saved model_predictions_per_train.csv")

files.download("model_comparison_per_train.csv")
if predictions:
    files.download("model_predictions_per_train.csv")



Top features for train 12314


,feature,importance
6,previous_delay,64.082100
7,delay_change,11.867649
15,day_of_year,6.946502
13,day,4.566750
17,previous_delay_used,2.800715
16,calculated_previous_delay,2.605657
10,day_of_week,2.429655
4,perivious_distance,1.041127
3,distance,1.031973
8,journey_progress,0.662907



Top features for train 12428


,feature,importance
6,previous_delay,56.960610
7,delay_change,31.750436
16,calculated_previous_delay,2.335246
17,previous_delay_used,1.908475
3,distance,1.832015
12,month,1.271273
4,perivious_distance,0.911638
15,day_of_year,0.903457
5,section_distance,0.550226
2,station_sequence,0.304231



Top features for train 12628


...

Saved model_comparison_per_train.csv
Saved model_predictions_per_train.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
import os

# Create a directory to store the models if it doesn't exist
model_dir = "catboost_models"
os.makedirs(model_dir, exist_ok=True)

for train_no, model in catboost_models.items():
    model_filename = os.path.join(model_dir, f"catboost_model_train_{train_no}.cbm")
    model.save_model(model_filename)
    print(f"Saved CatBoost model for train {train_no} to {model_filename}")

Saved CatBoost model for train 12314 to catboost_models/catboost_model_train_12314.cbm
Saved CatBoost model for train 12428 to catboost_models/catboost_model_train_12428.cbm
Saved CatBoost model for train 12628 to catboost_models/catboost_model_train_12628.cbm
Saved CatBoost model for train 12802 to catboost_models/catboost_model_train_12802.cbm
Saved CatBoost model for train 12958 to catboost_models/catboost_model_train_12958.cbm
Saved CatBoost model for train 14310 to catboost_models/catboost_model_train_14310.cbm
Saved CatBoost model for train 22457 to catboost_models/catboost_model_train_22457.cbm
Saved CatBoost model for train 22487 to catboost_models/catboost_model_train_22487.cbm


In [31]:
for train_no in catboost_models.keys():
    model_filename = os.path.join("catboost_models", f"catboost_model_train_{train_no}.cbm")
    files.download(model_filename)
    print(f"Downloaded CatBoost model for train {train_no}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded CatBoost model for train 12314


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Downloaded CatBoost model for train 22457


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded CatBoost model for train 22487


## Interpretation

**MAE** is the main metric to compare for delay prediction: lower is better.

**RMSE** is useful for seeing the impact of large mistakes: lower is better.

**R²** is supplementary: higher is better.

Each train has its own model, so the results compare CatBoost, LSTM and GRU independently for each train.

The target in this dataset is `final_delay`. This predicts delay in minutes; a true clock-time ETA additionally requires the scheduled arrival time.
